# <p style="font-family:newtimeroman;font-size:150%;text-align:center;color:#06445e;">Author: Mohammed Hamid </p>


# <p style="font-family:newtimeroman;font-size:110%;text-align:center;color:#06445e;">Phase - 1.2 Pandas: Topic - 15: Combining Datasets</p>


`merge (inner, left, right, outer); join keys and key uniqueness; validate= and indicator=; concat; many-to-many row explosion trap.`

### **Subtopic 1 — merge() and join types**
### **Intuition first**
Imagine you have two tables.

**Orders table**

| order_id | customer_id | quantity |
|---|---|---:|
| 101 | C01 | 2 |
| 102 | C02 | 1 |
| 103 | C03 | 4 |

**Customers table**

| customer_id | customer_name |
|---|---|
| C01 | Aisha |
| C02 | Rahul |
| C04 | Meera |

The orders table knows which customer placed each order, but it doesn't contain the customer's name.

The customers table contains names. We can combine the tables using their shared column, customer_id.

That shared column is called a join key.

### **Four common merge types**
1. Inner join -  Keeps only keys found in both tables.
2. Left join - Keeps every row from the left table; unmatched right-side values become missing.

3. Right join - Keeps every row from the right table.

4. Outer join - Keeps keys from both tables, including unmatched rows.

### **Your prediction exercise**
Using the two example tables above, predict the results before writing code.
1. For an inner merge, how many rows will remain?
2. For a left merge with Orders as the left table, how many rows will remain?
3. In that left merge, what will customer_name be for order 103?
4. For an outer merge, will customer C04 appear in the result?
5. Which column should we use as the join key?

1. only 2 rows. As it only keeps the rows those are present in both tables. 
2. only 3 rows.
3. NaN
4. Yes
5. customer_id

In [1]:
import pandas as pd
import numpy as np

table1 = pd.DataFrame({

    "order_id" : [101, 102, 103],
    "customer_id" : ["C01", "C02", "C03"],
    "quantity": [2, 1, 4]
    
    }
)

table2 = pd.DataFrame({
        "customer_id" : ["C01", "C02", "C04"],
        "customer_name": ["Aisha", "Rahul", "Meera"]
})


print(table1.merge(table2, how="outer"))

   order_id customer_id  quantity customer_name
0     101.0         C01       2.0         Aisha
1     102.0         C02       1.0         Rahul
2     103.0         C03       4.0           NaN
3       NaN         C04       NaN         Meera


### **Your task — Step 1**
In your notebook, write code to answer these questions:
1. What is the shape of both DataFrames?
2. How many missing values are there in customer_id in each DataFrame?
3. Does customer.csv contain duplicate customer_id values?
4. How many unique customer IDs are present in each DataFrame?
5. Are there order records whose customer_id does not exist in customer.csv?
Do not merge yet. First predict what you expect:
- If the customer table has 100 rows and 100 unique customer IDs, what does that suggest about its key?
- If an order contains a customer ID absent from the customer table, what information will be missing after a left merge?

In [2]:
orders = pd.read_csv("data/orders.csv")
customers= pd.read_csv("data/customers.csv")

print(f"Shape of DataFrame-1(order.csv): {orders.shape}")
print(f"Shape of DataFrame-2(customer.csv): {customers.shape}")

duplicate_customer_ids = customers["customer_id"].duplicated().sum()
print(f"Duplicate counts in DataFrame-2(customer.csv {duplicate_customer_ids}")
print(f"\nDuplicated id in customer.csv:\n{customers.loc[customers['customer_id'].duplicated(), ["customer_id", "customer_name"]]}\n")

orders_unique_customer_ids = orders['customer_id'].nunique()
customers_unique_customer_ids = customers["customer_id"].nunique()

print(f"Unique customer ids in orders: {(orders_unique_customer_ids)}")
print(f"Unique customer ids in customers: {(customers_unique_customer_ids)}")

missing_count_in_orders = orders["customer_id"].isna().sum()
missing_count_in_customers = customers["customer_id"].isna().sum()

print(f"Missing count in orders: {missing_count_in_orders}")
print(f"Missing count in customers : {missing_count_in_customers}")

Shape of DataFrame-1(order.csv): (500, 10)
Shape of DataFrame-2(customer.csv): (120, 6)
Duplicate counts in DataFrame-2(customer.csv 1

Duplicated id in customer.csv:
    customer_id customer_name
117       C0010    Diya Singh

Unique customer ids in orders: 116
Unique customer ids in customers: 119
Missing count in orders: 3
Missing count in customers : 0


In [3]:
print(customers.loc[customers["customer_id"] == "C0010", ["customer_id", "customer_name", "email"]],"\n")
# The id is same but the name and email are different.

orders_customer_ids = orders.loc[orders["customer_id"].notna(), ["customer_id"]]
customers_customer_ids = customers.loc[customers['customer_id'].notna(), "customer_id"]

print((orders_customer_ids.loc[~orders_customer_ids["customer_id"].isin(customers_customer_ids)]))

    customer_id customer_name                     email
9         C0010    Rohan Nair   rohannair10@example.com
117       C0010    Diya Singh  diyasingh118@example.com 

    customer_id
95        C0118
130       C0118
345       C0118
469       C0118
470       C9999


### **our next exercise**
Before resolving the conflicting customer record, let's quantify the risk.

Write code to find:
1. How many orders have customer_id == "C0010"?
2. What would the expected row count be after a left merge if each of those orders matched both customer records?
3. Why would that increase be dangerous for later calculations such as total sales or total quantity?
Then try a controlled merge using validate="many_to_one".

In [4]:
print((orders[orders['customer_id'] == "C0010"].shape[0]))
merged_orders_and_customers = orders.merge(customers, on="customer_id", how="left")
print(merged_orders_and_customers.head(2))

# I think it will make duplicate rows. 

2
  order_id customer_id product_id  order_date  quantity  unit_price  \
0   O00001       C0051       P009  2025-07-25       2.0      2499.0   
1   O00002       C0082       P010  2026-08-19       1.0       499.0   

   discount_pct payment_method order_status shipping_city customer_name  \
0          10.0            UPI    Delivered       Chennai     Isha Khan   
1          15.0           Card    Delivered        Mumbai   Aarav Singh   

                      email     city signup_date customer_segment  
0    ishakhan51@example.com  Chennai  2024-08-12          Regular  
1  aaravsingh82@example.com   Mumbai  2024-02-03          Regular  


### **2. Validate the merge**

In [5]:
print("Orders shape:", orders.shape)
print("Merged shape:", merged_orders_and_customers.shape)

print(
    merged_orders_and_customers.loc[
        merged_orders_and_customers["customer_id"] == "C0010",
        ["order_id", "customer_id", "customer_name"]
    ]
)

Orders shape: (500, 10)
Merged shape: (502, 15)
    order_id customer_id customer_name
77    O00078       C0010    Rohan Nair
78    O00078       C0010    Diya Singh
342   O00342       C0010    Rohan Nair
343   O00342       C0010    Diya Singh


In [6]:
merged_orders_and_customers = orders.merge(customers, on="customer_id", how="left", indicator=True)
merged_orders_and_customers

,order_id,customer_id,product_id,order_date,quantity,unit_price,discount_pct,payment_method,order_status,shipping_city,customer_name,email,city,signup_date,customer_segment,_merge
0,O00001,C0051,P009,2025-07-25,2.0,2499.00,10.0,UPI,Delivered,Chennai,Isha Khan,ishakhan51@example.com,Chennai,2024-08-12,Regular,both
1,O00002,C0082,P010,2026-08-19,1.0,499.00,15.0,Card,Delivered,Mumbai,Aarav Singh,aaravsingh82@example.com,Mumbai,2024-02-03,Regular,both
2,O00003,C0042,P017,2026-03-11,2.0,4249.15,0.0,Net Banking,Cancelled,Mumbai,Diya Shah,diyashah42@example.com,Mumbai,2025-04-14,Regular,both
3,O00004,C0120,P019,2026-08-14,1.0,1499.00,0.0,Net Banking,Delivered,Bengaluru,Rohan Gupta,rohangupta120@example.com,Bengaluru,2025-09-11,Premium,both
4,O00005,C0023,P009,2026-05-17,2.0,2499.00,15.0,UPI,Processing,Kolkata,Arjun Gupta,arjungupta23@example.com,Kolkata,2023-11-21,New,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
497,O00496,C0100,P013,2026-09-09,1.0,548.90,0.0,Net Banking,Returned,Jaipur,Meera Das,meeradas100@example.com,Jaipur,2025-07-16,Premium,both
498,O00497,C0017,P023,2025-04-03,4.0,1499.00,15.0,Cash,Returned,Kolkata,Sara Nair,saranair17@example.com,Kolkata,2024-01-12,New,both
499,O00498,C0049,P017,2025-02-19,4.0,5498.90,0.0,Card,Processing,Delhi,Rohan Das,rohandas49@example.com,Delhi,2024-09-18,Premium,both
500,O00499,C0095,P001,2026-10-05,2.0,199.00,10.0,Net Banking,Delivered,Chennai,Rohan Mehta,rohanmehta95@example.com,Chennai,2024-09-17,New,both


### **Your next validation task**
Use the _merge column to calculate:
1. Number of both rows.
2. Number of left_only rows.
3. Number of unmatched unique customer IDs among left_only rows.

In [7]:
both_count = merged_orders_and_customers[merged_orders_and_customers['_merge'] == "both"]
left_count = merged_orders_and_customers[merged_orders_and_customers['_merge'] == "left_only"]
print(both_count.shape[0])
print(left_count.shape[0])

494
8


### **Your next small task**
Find the unique customer IDs among the left_only rows.

Use your existing left_count DataFrame and return the unique values from its customer_id column. Then tell me:
- How many unmatched rows?
- How many unmatched unique IDs?

In [8]:
unique_left = left_count["customer_id"].dropna().unique()

print(f"Unmatched rows: {left_count.shape[0]}")
print(f"Unmatched unique IDs: {len(unique_left)}")
print(unique_left)

Unmatched rows: 8
Unmatched unique IDs: 2
<StringArray>
['C0118', 'C9999']
Length: 2, dtype: str


### **What you have learned so far**
You have practised:
- Choosing a join key
- Checking missing and duplicate keys
- Identifying conflicting duplicate records
- Understanding many-to-one relationships
- Detecting row multiplication
- Using validate="many_to_one" to catch invalid relationships
- Using indicator=True to find unmatched records
- Reconciling merge row counts

### **Subtopic 2: concat()**
### **1. Intuition: merge() vs concat()**

### **1. merge()**: 
Match records using a key.

Example: attach customer names to orders using customer_id.

### **2. concat()**:
Put tables together along an axis.

Example: append January orders and February orders into one table.


### **The main idea:**
- merge() connects related information.
- concat() combines pieces of a dataset.

For concat(), the most common operation is vertical concatenation: adding rows.

In [12]:
orders_part1 = orders.iloc[:250]
orders_part2 = orders.iloc[250:]
orders_part2.shape

(250, 10)

Before writing the concatenation code, predict:
1. What is the shape of orders_part1?
2. What is the shape of orders_part2?
3. If we stack them vertically, how many rows should the combined DataFrame contain?
4. How many columns should it contain?
5. Does concat() need to match customer_id values, like merge() does?

### **Answer**

1. (250, 10)
2. (250, 10)
3. 500 rows
4. 10 columns
5. No. There is no need in concat to match customer_id values like merge().

### **Exercise 1 — Perform vertical concatenation**

Now write the code yourself to combine orders_part1 and orders_part2 vertically using pd.concat().

In [14]:
combined_orders = pd.concat([orders_part1, orders_part2])


After running it, validate:
1. Shape should be (500, 10).
2. Compare the combined row count with the original.
3. Check whether order_id duplicates exist in the combined DataFrame.
4. Confirm that the combined data still contains the expected columns.

In [24]:
print(f"Combined orders shape: {combined_orders.shape}")
print(orders.shape == combined_orders.shape)

print(f"Duplicate counts in original orders: {orders.loc[orders.duplicated(), ["customer_id", "order_id"]]}")
# Customer C0096 is duplicate

print(f"Duplicate counts in combined orders:\n{combined_orders.loc[combined_orders.duplicated(), ["customer_id", "order_id"]]}")
# There is only one duplicate value in combined orders.

print(f"Expected combined columns: {combined_orders.columns.tolist()}")

Combined orders shape: (500, 10)
True
Duplicate counts in original orders:     customer_id order_id
499       C0096   O00121
Duplicate counts in combined orders:
    customer_id order_id
499       C0096   O00121
Expected combined columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'order_status', 'shipping_city']


In [25]:
orders.equals(combined_orders)

True

### **Subtopic: Horizontal concat()**

1. Intuition
Imagine two tables containing information about the same 3 orders.

| order_id | quantity |
|---|---:|
| O001 | 2 |
| O002 | 5 |
| O003 | 1 |



| order_id | note |
|---|---|
| O001 | Gift |
| O002 | Urgent |
| O003 | Regular |


Horizontal concatenation places their columns beside each other:
| order_id | quantity | note |
|---|---:|---|
| O001 | 2 | Gift |
| O002 | 5 | Urgent |
| O003 | 1 | Regular |


This is called horizontal concatenation.

In [ ]:
pd.concat([orders_part1, orders_part2], axis=1)  # axis=1 means horizontal concatenate.


### **2. Important difference from merge()**
Horizontal concat() aligns rows using their index labels, not by matching a column such as order_id.

If the indexes do not align, values may appear beside the wrong records or become missing.

### **Your prediction exercise**

In [30]:
order_ids = orders[['order_id']]
quantities = orders[["quantity"]]

combined_df = pd.concat([order_ids, quantities], axis=1)
print(combined_df.shape)

(500, 2)


### **Predict:**
1. What is the shape of order_ids?
2. What is the shape of quantities?
3. What will the shape be after horizontal concatenation?
4. Does axis=1 stack rows or place columns side by side?
5. If both DataFrames retain their original index, which index does concat() use to align the rows?

1. (500, 1)
2. (500, 1)
3. (500, 2)
4. It place columns side by side.
5. It will use combined index labels


### **Exercise — Perform horizontal concatenation**
Now write the code yourself:
- Combine order_ids and quantities using pd.concat().
- Use axis=1.
Then validate the result:
1. Shape is (500, 2).
2. Column names are order_id and quantity.
3. Compare the combined columns with the original columns using .equals().
4. Check that the resulting index matches the original orders index.

In [42]:
combined_order_id_and_quantity = pd.concat([order_ids, quantities], axis=1)
print(f"Shape: {combined_order_id_and_quantity.shape}")
print(f"Column names: {combined_order_id_and_quantity.columns.tolist()}")
print(orders[["order_id", "quantity"]].equals(combined_order_id_and_quantity))
print(orders.index.equals(combined_order_id_and_quantity.index))

Shape: (500, 2)
Column names: ['order_id', 'quantity']
True
True
